# Plant Disease Detection CNN - Training
## Using Real PlantVillage Dataset

Train a CNN model on the downloaded PlantVillage dataset (39 disease classes)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pickle

print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {keras.__version__}")
print(f"NumPy: {np.__version__}")

%matplotlib inline


In [ ]:
# Load dataset from PlantVillage folder structure
DATASET_PATH = 'Plant Village Dataset'
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 50

print("="*60)
print("LOADING PLANTVILLAGE DATASET")
print("="*60)

# Verify dataset exists
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"Dataset not found at {DATASET_PATH}")

# Get all disease classes from Train folder
train_path = os.path.join(DATASET_PATH, 'Train')
disease_classes = sorted([d for d in os.listdir(train_path) if os.path.isdir(os.path.join(train_path, d))])

print(f"\nDataset found at: {DATASET_PATH}")
print(f"Number of disease classes: {len(disease_classes)}")
print(f"\nDisease classes:")
for i, disease in enumerate(disease_classes, 1):
    train_count = len(os.listdir(os.path.join(train_path, disease)))
    print(f"  {i:2d}. {disease:40s} - {train_count} images")

NUM_CLASSES = len(disease_classes)


In [ ]:
# Setup data generators with ADVANCED augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,  # Increased from 20 for more robust rotation invariance
    width_shift_range=0.3,  # Increased from 0.2 for more position variations
    height_shift_range=0.3,  # Increased from 0.2
    shear_range=0.3,  # Increased from 0.2 for more shear variations
    zoom_range=0.3,  # Increased from 0.2 for zoom variations
    horizontal_flip=True,
    vertical_flip=True,  # Added for more diversity
    brightness_range=[0.8, 1.2],  # Added for lighting condition variations
    fill_mode='reflect',  # Changed from 'nearest' for better edge handling
    channel_shift_range=20.0  # Added for color/lighting variations
)

val_test_datagen = ImageDataGenerator(rescale=1./255)

print("\nLoading training data...")
train_generator = train_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, 'Train'),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

print("Loading validation data...")
val_generator = val_test_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, 'Val'),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("Loading test data...")
test_generator = val_test_datagen.flow_from_directory(
    os.path.join(DATASET_PATH, 'Test'),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"\n✓ Data loaded successfully!")
print(f"  Training batches: {train_generator.samples // BATCH_SIZE}")
print(f"  Validation batches: {val_generator.samples // BATCH_SIZE}")
print(f"  Test batches: {test_generator.samples // BATCH_SIZE}")

# Save class indices mapping
class_indices = train_generator.class_indices
print(f"  Total classes: {len(class_indices)}")

In [ ]:
# Build CNN Model using Transfer Learning (MobileNetV2)
print("\n" + "="*60)
print("BUILDING CNN MODEL")
print("="*60)

# Use pre-trained MobileNetV2 for better performance
base_model = keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze base model weights initially
base_model.trainable = False

# Add custom top layers
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

print("\nModel Architecture:")
model.summary()

# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n✓ Model compiled successfully!")


In [ ]:
# Build CNN Model using Transfer Learning (MobileNetV2)
print("\n" + "="*60)
print("BUILDING CNN MODEL WITH TRANSFER LEARNING")
print("="*60)

# Use pre-trained MobileNetV2 for better performance
base_model = keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze base model weights initially
base_model.trainable = False

# Add custom top layers with improved architecture for better predictions
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),  # Increased capacity from 256
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

print("\nModel Architecture:")
model.summary()

# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n✓ Model compiled successfully!")


In [ ]:
# Train the model
print("\n" + "="*60)
print("TRAINING MODEL")
print("="*60)

# Define callbacks
os.makedirs('models', exist_ok=True)

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        'models/best_disease_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# Train phase 1: frozen base model
print("\nPhase 1: Training top layers (base model frozen)...")
history1 = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

# Unfreeze last layers of base model
print("\nPhase 2: Fine-tuning last layers of base model...")
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print("\n✓ Training completed!")


In [ ]:
# Train the model
print("\n" + "="*60)
print("TRAINING MODEL WITH ADVANCED TRANSFER LEARNING")
print("="*60)

# Define callbacks with improved settings
os.makedirs('models', exist_ok=True)

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=8,  # Increased patience for better convergence
        restore_best_weights=True,
        verbose=1,
        min_delta=0.001  # Minimum change threshold
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-8,  # Very low minimum learning rate
        verbose=1
    ),
    ModelCheckpoint(
        'models/best_disease_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1,
        save_weights_only=False  # Save complete model
    )
]

# Train phase 1: frozen base model with data augmentation
print("\n" + "="*60)
print("PHASE 1: Training top layers (Base model frozen)")
print("="*60)
print("This phase focuses on learning disease patterns with a frozen pre-trained base.\n")

history1 = model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1,
    class_weight='auto'  # Handle class imbalance automatically
)

# Unfreeze more layers of base model for fine-tuning
print("\n" + "="*60)
print("PHASE 2: Fine-tuning last 100 layers of MobileNetV2")
print("="*60)
print("This phase fine-tunes the base model for better real-world predictions.\n")

base_model.trainable = True
for layer in base_model.layers[:-100]:  # Keep first layers frozen
    layer.trainable = False

# Verify which layers are trainable
trainable_count = sum([1 for layer in base_model.layers if layer.trainable])
print(f"Fine-tuning {trainable_count} layers out of {len(base_model.layers)}")

# Use much lower learning rate for fine-tuning
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.00005),  # Very low learning rate
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_generator,
    epochs=15,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1,
    class_weight='auto'
)

print("\n✓ Training completed successfully!")
print("✓ Best model saved to: models/best_disease_model.h5")


In [ ]:
# Visualize results
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Training history - accuracy
axes[0, 0].plot(history1.history['accuracy'], label='Phase 1 Train', linewidth=2)
axes[0, 0].plot(history1.history['val_accuracy'], label='Phase 1 Val', linewidth=2)
axes[0, 0].plot([len(history1.history['accuracy']) + i for i in range(len(history2.history['accuracy']))],
                history2.history['accuracy'], label='Phase 2 Train', linewidth=2)
axes[0, 0].plot([len(history1.history['val_accuracy']) + i for i in range(len(history2.history['val_accuracy']))],
                history2.history['val_accuracy'], label='Phase 2 Val', linewidth=2)
axes[0, 0].set_title('Model Accuracy', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Training history - loss
axes[0, 1].plot(history1.history['loss'], label='Phase 1 Train', linewidth=2)
axes[0, 1].plot(history1.history['val_loss'], label='Phase 1 Val', linewidth=2)
axes[0, 1].plot([len(history1.history['loss']) + i for i in range(len(history2.history['loss']))],
                history2.history['loss'], label='Phase 2 Train', linewidth=2)
axes[0, 1].plot([len(history1.history['val_loss']) + i for i in range(len(history2.history['val_loss']))],
                history2.history['val_loss'], label='Phase 2 Val', linewidth=2)
axes[0, 1].set_title('Model Loss', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Confusion Matrix (for a subset due to large number of classes)
# Use only top 15 classes for visualization
if len(disease_classes) > 15:
    top_indices = y_true[:1000]
    cm_subset = cm[np.unique(top_indices)][:, np.unique(top_indices)]
    classes_subset = [disease_classes[i] for i in np.unique(top_indices)[:15]]
    sns.heatmap(cm_subset[:15, :15], annot=False, fmt='d', cmap='Blues', ax=axes[1, 0], cbar=True)
    axes[1, 0].set_title('Confusion Matrix (Top 15 Classes)', fontsize=12, fontweight='bold')
else:
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', ax=axes[1, 0])
    axes[1, 0].set_title('Confusion Matrix', fontsize=12, fontweight='bold')

axes[1, 0].set_ylabel('True Label')
axes[1, 0].set_xlabel('Predicted Label')

# Plot 4: Per-class accuracy
per_class_acc = cm.diagonal() / cm.sum(axis=1)
top_15_indices = np.argsort(per_class_acc)[-15:]
top_diseases = [disease_classes[i] for i in top_15_indices]
top_accs = per_class_acc[top_15_indices]

axes[1, 1].barh(range(len(top_diseases)), top_accs, color='skyblue', edgecolor='navy')
axes[1, 1].set_yticks(range(len(top_diseases)))
axes[1, 1].set_yticklabels(top_diseases, fontsize=9)
axes[1, 1].set_xlabel('Accuracy')
axes[1, 1].set_title('Top 15 Classes by Accuracy', fontsize=12, fontweight='bold')
axes[1, 1].set_xlim([0, 1])

for i, v in enumerate(top_accs):
    axes[1, 1].text(v + 0.02, i, f'{v:.2%}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print("✓ Visualization complete!")


In [ ]:
# Save the model
print("\n" + "="*60)
print("SAVING MODEL")
print("="*60)

# Save model in multiple formats
model.save('models/plant_disease_model.h5')
model.save('models/plant_disease_model_full/')

# Save disease classes
with open('models/disease_classes.pkl', 'wb') as f:
    pickle.dump(disease_classes, f)

# Save class indices
with open('models/class_indices.pkl', 'wb') as f:
    pickle.dump(class_indices, f)

# Save model info
import json
model_info = {
    'model_name': 'PlantVillage Disease Detection',
    'num_classes': NUM_CLASSES,
    'classes': disease_classes,
    'test_accuracy': float(test_accuracy),
    'test_loss': float(test_loss),
    'image_size': IMAGE_SIZE,
    'architecture': 'MobileNetV2 with Transfer Learning'
}

with open('models/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=4)

print("\n✓ Model saved successfully!")
print(f"\nSaved files:")
print(f"  - models/plant_disease_model.h5 (Keras format)")
print(f"  - models/plant_disease_model_full/ (SavedModel format)")
print(f"  - models/disease_classes.pkl (Class names)")
print(f"  - models/class_indices.pkl (Class indices)")
print(f"  - models/model_info.json (Model metadata)")
print(f"  - models/best_disease_model.h5 (Best checkpoint)")

print(f"\n" + "="*60)
print(f"TRAINING COMPLETE")
print(f"="*60)
print(f"Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Number of Classes: {NUM_CLASSES}")
print(f"Model Parameters: {model.count_params():,}")
print(f"="*60)


In [ ]:
# Make predictions on new images
def predict_disease(image_path, model, class_names):
    """Predict disease for a single image"""
    
    # Load and preprocess image
    img = load_img(image_path, target_size=IMAGE_SIZE)
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)
    
    # Get prediction
    prediction = model.predict(img_array, verbose=0)
    confidence = np.max(prediction)
    predicted_idx = np.argmax(prediction)
    predicted_disease = class_names[predicted_idx]
    
    # Get top 3 predictions
    top_3_indices = np.argsort(prediction[0])[::-1][:3]
    top_3 = [
        {
            'disease': class_names[idx],
            'confidence': f"{prediction[0][idx]*100:.2f}%"
        }
        for idx in top_3_indices
    ]
    
    return {
        'predicted_disease': predicted_disease,
        'confidence': f"{confidence*100:.2f}%",
        'top_3': top_3
    }

# Test on a few images from test set
print("\n" + "="*60)
print("SAMPLE PREDICTIONS")
print("="*60)

# Get sample test images
test_path = os.path.join(DATASET_PATH, 'Test')
sample_predictions = []

for disease_folder in disease_classes[:5]:
    disease_path = os.path.join(test_path, disease_folder)
    if os.path.exists(disease_path):
        images = os.listdir(disease_path)
        if len(images) > 0:
            img_file = os.path.join(disease_path, images[0])
            result = predict_disease(img_file, model, disease_classes)
            
            print(f"\nTest: {disease_folder}")
            print(f"  Predicted: {result['predicted_disease']}")
            print(f"  Confidence: {result['confidence']}")
            print(f"  Top 3:")
            for pred in result['top_3']:
                print(f"    - {pred['disease']}: {pred['confidence']}")

print("\n✓ Sample predictions complete!")
